# VisionVault (Images Only)
Global semantic search for images using Gemini embeddings + ChromaDB. Database lives in `~/.visionvault`.

In [ ]:
# ─────────────────────────────────────────────
# STEP 1 — Install dependencies
# ─────────────────────────────────────────────
!pip install -r requirements.txt

In [ ]:
# ─────────────────────────────────────────────
# STEP 2 — Imports & configuration
# ─────────────────────────────────────────────
import os
import time
import base64
import hashlib
from io import BytesIO
from pathlib import Path

import numpy as np
import requests
import chromadb
from PIL import Image

# --- API ---
API_KEY  = os.environ.get("GEMINI_API_KEY", "")   # set in your environment, never hardcode
MODEL    = "models/gemini-embedding-2-preview"
BASE_URL = "https://generativelanguage.googleapis.com/v1beta"
DIM      = 768

# --- Supported image formats ---
IMG_EXT  = {".png", ".jpg", ".jpeg", ".webp", ".gif", ".bmp"}
MIME_MAP = {
    ".png":  "image/png",
    ".jpg":  "image/jpeg",
    ".jpeg": "image/jpeg",
    ".webp": "image/webp",
    ".gif":  "image/gif",
    ".bmp":  "image/bmp",
}

# --- Database & image settings ---
DB_DIR         = Path.home() / ".visionvault"   # stored in home dir, outside this repo
MAX_IMAGE_SIDE = 1024                           # resize before embedding to keep payloads small

In [ ]:
# ─────────────────────────────────────────────
# STEP 3 — Gemini API helpers
# ─────────────────────────────────────────────

def _post(url, body):
    """POST to Gemini API with automatic retry on rate-limit / server errors."""
    if not API_KEY:
        raise RuntimeError(
            "Missing GEMINI_API_KEY. Set it as an environment variable before running."
        )
    hdrs = {"Content-Type": "application/json", "x-goog-api-key": API_KEY}
    for i in range(3):
        r = requests.post(url, json=body, headers=hdrs, timeout=180)
        if r.status_code == 429 or r.status_code >= 500:
            time.sleep(float(r.headers.get("Retry-After", 2 ** i)))
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError("API failed after 3 retries.")


def _norm(v):
    """L2-normalise a vector so cosine similarity works correctly."""
    a = np.array(v, dtype=np.float32)
    n = np.linalg.norm(a)
    return a / n if n else a


def embed_query(query: str) -> np.ndarray:
    """Embed a text search query."""
    body = {
        "requests": [
            {
                "model": MODEL,
                "content": {"parts": [{"text": query}]},
                "taskType": "RETRIEVAL_QUERY",
                "outputDimensionality": DIM,
            }
        ]
    }
    data = _post(f"{BASE_URL}/{MODEL}:batchEmbedContents", body)
    return _norm(data["embeddings"][0]["values"])


def _prepare_image_bytes(path: Path, mime: str) -> tuple[bytes, str]:
    """Resize image and return (bytes, mime_type). GIFs are kept as-is."""
    if mime == "image/gif":
        return path.read_bytes(), mime
    try:
        with Image.open(path) as im:
            im = im.convert("RGB")
            im.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE))
            buf = BytesIO()
            im.save(buf, format="JPEG", quality=90, optimize=True)
            return buf.getvalue(), "image/jpeg"
    except Exception:
        return path.read_bytes(), mime   # fallback: raw bytes


def embed_image(path: Path, mime: str) -> np.ndarray:
    """Embed a local image file."""
    data_bytes, out_mime = _prepare_image_bytes(path, mime)
    b64 = base64.standard_b64encode(data_bytes).decode()
    body = {
        "model": MODEL,
        "outputDimensionality": DIM,
        "content": {
            "parts": [{"inline_data": {"mime_type": out_mime, "data": b64}}]
        },
    }
    return _norm(_post(f"{BASE_URL}/{MODEL}:embedContent", body)["embedding"]["values"])

In [ ]:
# ─────────────────────────────────────────────
# STEP 4 — ChromaDB vector store  (maximum privacy)
# Stores only hashed IDs + non-sensitive metadata.
# No absolute file paths are ever saved.
# ─────────────────────────────────────────────

class Store:
    def __init__(self):
        DB_DIR.mkdir(parents=True, exist_ok=True)
        client   = chromadb.PersistentClient(path=str(DB_DIR))
        self.col = client.get_or_create_collection(
            "visionvault_images",
            metadata={"hnsw:space": "cosine"}
        )

    def put(self, file_id: str, vec: np.ndarray, doc: str, meta: dict):
        """Insert or update one record."""
        self.col.upsert(
            ids=[file_id],
            embeddings=[vec.tolist()],
            documents=[doc],
            metadatas=[meta]
        )

    def find(self, vec: np.ndarray, k: int = 20) -> list:
        """Return top-k nearest neighbours."""
        n = min(k, self.col.count() or 1)
        if n == 0:
            return []
        r = self.col.query(
            query_embeddings=[vec.tolist()],
            n_results=n,
            include=["documents", "metadatas", "distances"],
        )
        if not r["ids"][0]:
            return []
        return [
            {
                "id":    r["ids"][0][i],
                "score": 1 - r["distances"][0][i],
                "name": (r.get("metadatas", [[{}]])[0][i] or {}).get("name"),
            }
            for i in range(len(r["ids"][0]))
        ]

    def ids(self) -> set:
        """Return all stored IDs (for deduplication)."""
        if not self.col.count():
            return set()
        return set(self.col.get(include=[]).get("ids", []))

In [ ]:
# ─────────────────────────────────────────────
# STEP 5 — Indexer  (maximum privacy)
# File identity = SHA-256 hash of contents.
# No absolute paths are stored anywhere.
# ─────────────────────────────────────────────

def sha(p: Path) -> str:
    """Return SHA-256 hex digest of a file (used as privacy-safe ID)."""
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


def index_dir(root):
    """
    Recursively index all supported images under `root`.
    Already-indexed images are skipped automatically.
    """
    root  = Path(root).resolve()
    store = Store()
    existing = store.ids()

    # Collect image files, skip hidden directories
    files = sorted(
        p for p in root.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMG_EXT
        and not any(part.startswith(".") for part in p.relative_to(root).parts)
    )

    # Filter out already-indexed files
    todo = [(f, sha(f)) for f in files if sha(f) not in existing]

    if not todo:
        print("✅ All images are already indexed.")
        return

    print(f"📂 {len(todo)} new image(s) to index...")

    for f, file_id in todo:
        mime = MIME_MAP.get(f.suffix.lower(), "application/octet-stream")
        print(f"  → {f.name}")
        try:
            vec = embed_image(f, mime)
            store.put(
                file_id,
                vec,
                f"[{mime}] {f.name}",
                {"name": f.name, "mime": mime},
            )
        except Exception as e:
            print(f"  ⚠️  Skipped {f.name}: {e}")

    print(f"\n✅ Done — {len(todo)} image(s) indexed.")

In [ ]:
# ─────────────────────────────────────────────
# STEP 6 — Search
# Results contain only: score, filename, hash ID.
# No absolute paths are ever returned.
# ─────────────────────────────────────────────

def search_images(query: str, k: int = 5) -> list:
    """
    Search indexed images with a natural language query.
    Returns up to `k` results with relevance score >= 0.15.
    """
    store = Store()
    if not store.col.count():
        print("⚠️  No images indexed yet. Run index_dir() first.")
        return []

    qvec = embed_query(query)
    hits = store.find(qvec, k=k)
    return [h for h in hits if h["score"] >= 0.15]

---
## ▶️ Run It

In [ ]:
# ── Index a folder ──────────────────────────
# Uncomment and edit the path for your system:

# macOS / Linux:
# index_dir("/Users/yourname/Pictures")

# Windows:
# index_dir(r"D:\Pictures")

In [ ]:
# ── Search your image database ───────────────
# Maximum privacy: results do NOT contain absolute paths.

results = search_images("sunglasses", k=3)

for i, r in enumerate(results, 1):
    label = r.get("name") or r["id"]
    print(f"{i}. [{r['score']:.3f}] {label}")
    print(f"       id = {r['id']}")